# Fooocus 2.5.6: arranque funcional para Colab

Este notebook prioriza **funcionar primero**: valida la GPU, descarga y comprueba Juggernaut XL con aria2, inicia Fooocus localmente y publica la UI mediante Cloudflare en vez de `gradio.live`.

Antes de ejecutar: **Entorno de ejecución > Cambiar tipo de entorno > T4 GPU**.

In [ ]:
# @title 1. Preparar repositorio y modelos
RAMA = 'fix/colab-functional-bootstrap'  # @param {type:"string"}
CACHEAR_MODELOS_EN_DRIVE = False  # @param {type:"boolean"}
EXTRAS_OPCIONALES = False  # @param {type:"boolean"}
MODO_ALTA_VRAM = False  # @param {type:"boolean"}

import os, shutil, subprocess, sys

REPO = 'https://github.com/deleonramiro085/Fooocus.git'
WORKDIR = '/content/Fooocus'

if not os.path.isdir(os.path.join(WORKDIR, '.git')):
    shutil.rmtree(WORKDIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', RAMA, REPO, WORKDIR], check=True)
else:
    subprocess.run(['git', '-C', WORKDIR, 'fetch', 'origin', RAMA], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'checkout', '-B', RAMA, f'origin/{RAMA}'], check=True)

if CACHEAR_MODELOS_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    cache = '/content/drive/MyDrive/Fooocus/models'
    for sub in ('checkpoints', 'inpaint', 'prompt_expansion', 'vae_approx', 'loras', 'vae'):
        target = os.path.join(cache, sub)
        local = os.path.join(WORKDIR, 'models', sub)
        os.makedirs(target, exist_ok=True)
        if not os.path.islink(local):
            shutil.rmtree(local, ignore_errors=True)
            os.symlink(target, local)
    print('Caché persistente:', cache)

cmd = [sys.executable, '-u', os.path.join(WORKDIR, 'colab_launcher.py')]
if EXTRAS_OPCIONALES: cmd.append('--extras')
if MODO_ALTA_VRAM: cmd.append('--high-vram')
print('Ejecutando:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## Confirmación funcional

Cuando aparezca `FOOOCUS LISTO`, abre la URL de Cloudflare. Genera **1 imagen** con el prompt `a red fox astronaut, cinematic photo`. La prueba queda confirmada cuando la galería muestra la imagen y el registro imprime `Total time`.

No actives extras ni alta VRAM durante esta primera validación. Si falla, copia desde la primera línea `[Env]` hasta el error final.

In [ ]:
# @title 2. Diagnóstico rápido (solo si falla)
import glob, os, platform, subprocess
print('python', platform.python_version())
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=False)
for path in glob.glob('/content/Fooocus/models/checkpoints/*'):
    print(os.path.basename(path), f'{os.path.getsize(path)/(1024**3):.2f} GB')
